In [2]:
import pandas as pd
import glob
import re

# ใช้ glob กวาดหาไฟล์ CSV ทั้งหมดที่ชื่อขึ้นต้นด้วย "สถิติจำนวนประชากรและบ้าน"
files = glob.glob("สถิติจำนวนประชากรและบ้าน*.csv")

# เตรียม List ของเดือน 12 เดือน สำหรับนำไปใช้ขยายร่างข้อมูล (Broadcasting)
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
results = []

# วนลูปอ่านข้อมูลและสกัดปี พ.ศ. จากชื่อไฟล์
for file in files:
    # ใช้ Regex ดึงตัวเลขปี พ.ศ. (2563 ถึง 2567) ออกจากชื่อไฟล์
    year_match = re.search(r'256[3-7]', file)
    if not year_match: continue

    # Data Transformation: แปลงปี พ.ศ. เป็น ค.ศ. (ลบ 543)
    # เพื่อให้แกนเวลา ตรงกับตารางฝุ่น PM2.5 และตารางอื่นๆ
    year_en = int(year_match.group()) - 543

    # อ่านไฟล์ CSV โดยข้าม 3 บรรทัดแรก (header=3)
    # เหตุผล: ไฟล์สถิติของรัฐมักมีหัวตารางที่ Merge Cell หรือเป็นชื่อรายงานซ่อนอยู่ด้านบน
    df = pd.read_csv(file, header=3)

    # กระบวนการทำความสะอาดข้อมูล
    for idx, row in df.iterrows():
        # ลบช่องว่าง (Whitespace) หน้าและหลังชื่อพื้นที่
        prov = str(row['พื้นที่']).strip()

        # ลบลูกน้ำ (Comma) ออกจากตัวเลขประชากร เพื่อเตรียมแปลงเป็นตัวเลขทางคณิตศาสตร์
        pop_str = str(row['รวม (รวม)']).strip().replace(',', '')

        # กรองข้อมูลขยะ: ข้ามบรรทัดที่ว่างเปล่า (nan) หรือบรรทัดที่เป็นยอดรวมระดับประเทศ/ภูมิภาค
        if prov == 'nan' or 'รวม' in prov or prov == '': continue

        # Data Standardization: ลบคำว่า "จังหวัด" ออกจากชื่อพื้นที่
        # เหตุผล: ตารางอื่น (เช่น PM2.5) ใช้ชื่อตัวจังหวัดโดดๆ
        # หากไม่ลบคำนี้ออก ตอนนำตารางไป Join กันข้อมูลจะหลุด ทันที
        if prov.startswith('จังหวัด'):
            prov = prov.replace('จังหวัด', '').strip()

        # แปลงข้อมูลประชากรจาก String ให้เป็น Integer
        try: pop_val = int(pop_str)
        except: continue

        # การขยายข้อมูลรายปีเป็นรายเดือน
        # ปัญหา: ข้อมูลประชากรสำรวจแค่ปีละ 1 ครั้ง แต่เราต้องการวิเคราะห์ฝุ่นเป็นรายเดือน
        # วิธีแก้: นำยอดประชากรของปีนั้นๆ ไปใส่ซ้ำ ให้ครบทั้ง 12 เดือน
        # (สมมติฐาน: จำนวนประชากรในแต่ละเดือนภายในปีเดียวกันไม่มีการเปลี่ยนแปลงอย่างมีนัยสำคัญ)
        for m in months:
            results.append({
                'Year': year_en,
                'Month': m,
                'Province': prov,
                'Total_Population': pop_val
            })

df_pop = pd.DataFrame(results)

# บันทึกเป็นไฟล์ CSV พร้อมตั้งค่า utf-8-sig เพื่อให้อ่านภาษาไทยได้สมบูรณ์
df_pop.to_csv('Cleaned_Population_Monthly.csv', index=False, encoding='utf-8-sig')

# ตรวจสอบความถูกต้อง: ต้องได้ 4,620 แถวพอดีเป๊ะ (77 จังหวัด x 5 ปี x 12 เดือน)
print(f"จำนวนทั้งหมด: {len(df_pop)} แถว (77 จังหวัด x 5 ปี x 12 เดือน)")
display(df_pop.head())

จำนวนทั้งหมด: 4620 แถว (77 จังหวัด x 5 ปี x 12 เดือน)


,Year,Month,Province,Total_Population
0,2020,Jan,กรุงเทพมหานคร,5588222
1,2020,Feb,กรุงเทพมหานคร,5588222
2,2020,Mar,กรุงเทพมหานคร,5588222
3,2020,Apr,กรุงเทพมหานคร,5588222
4,2020,May,กรุงเทพมหานคร,5588222
